<a href="https://colab.research.google.com/github/Proton2-Limitless/RL/blob/main/Copy_of_ann.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Artificial Neural Network

### Importing the libraries

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf

In [ ]:
tf.__version__

'2.20.0'

## Part 1 - Data Preprocessing

### Importing the dataset

In [ ]:
dataset = pd.read_csv('/Churn_Modelling.csv')
X = dataset.iloc[:, 3:-1].values #all columns from index 3 to last -1
y = dataset.iloc[:, -1].values #only last value, our dependent variable, output

In [ ]:
print(X)

[[619 'France' 'Female' ... 1 1 101348.88]
 [608 'Spain' 'Female' ... 0 1 112542.58]
 [502 'France' 'Female' ... 1 0 113931.57]
 ...
 [709 'France' 'Female' ... 0 1 42085.58]
 [772 'Germany' 'Male' ... 1 0 92888.52]
 [792 'France' 'Female' ... 1 0 38190.78]]


In [ ]:
print(y)

[1 0 1 ... 1 1 0]


### Encoding categorical data

Label Encoding the "Gender" column

In [ ]:
from sklearn.preprocessing import LabelEncoder #label encoder for converting text to integers as machine dont
                                              #understand texts
le = LabelEncoder() #tool initializing
X[:, 2] = le.fit_transform(X[:, 2]) #where mathematical transformation occur, this target all rows and specifically
                                    #column 2(Gender) of the new sliced data,
                                    #.fit_transform(...), This does two jobs at once, a. fit: It looks at the column
                                    # and learns all the unique categories (it finds "Female" and "Male").
                                    # b. transform: It replaces the text with numbers based on alphabetical order.
                                    #X[:, 2] = ...: This takes those new numbers and overwrites the old text values
                                    # directly inside your matrix $X$

One Hot Encoding the "Geography" column

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# 1. Define the transformation rules
# We name this step 'encoder', tell it to use OneHotEncoder(), and target column index 1 (Geography).
# remainder='passthrough' ensures we don't drop the rest of the numerical columns in X.
ct = ColumnTransformer(
    transformers=[("encoder", OneHotEncoder(), [1])], remainder="passthrough"
)

# 2. Apply the transformation and overwrite X
# fit_transform learns the unique countries and converts them into 1s and 0s.
# np.array() ensures the final output is saved as a clean NumPy array.
X = np.array(ct.fit_transform(X))

### Splitting the dataset into the Training set and Test set

In [ ]:
from sklearn.model_selection import train_test_split

# Split the dataset into training sets (80%) and testing sets (20%)
# random_state=0 ensures the random split is identical every time you run the cell
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0
)

# After this line:
# Use X_train and y_train to TRAIN your machine learning model.
# Use X_test and y_test to TEST and evaluate its final accuracy.

### Feature Scaling

In [ ]:
#Because numbers like 150,000 are vastly larger than 40 or 1, a machine learning model
#will look at the mathematical equations and mistakenly think Salary is thousands of times more important than Age.

#StandardScaler squishes all these columns down so they are on the same scale, centering them around 0 with a
#standard deviation of 1. This forces the model to treat all features fairly.

from sklearn.preprocessing import StandardScaler

# 1. Initialize the Standarization tool
sc = StandardScaler()

# 2. Scale the Training features
# fit_transform calculates the mean/variance AND transforms the training data
X_train = sc.fit_transform(X_train)

# 3. Scale the Testing features
# transform ONLY scales the data using the mean/variance already learned from X_train
X_test = sc.transform(X_test)

## Part 2 - Building the ANN

### Initializing the ANN

In [ ]:
# 1. Initialize the ANN as a sequence of layers
ann = tf.keras.models.Sequential()

### Adding the input layer and the first hidden layer

In [ ]:
# 2. Add the first fully-connected hidden layer (6 neurons, ReLU activation)
ann.add(tf.keras.layers.Dense(units=6, activation="relu"))

### Adding the second hidden layer

In [ ]:
# 3. Add the second fully-connected hidden layer (6 neurons, ReLU activation)
ann.add(tf.keras.layers.Dense(units=6, activation="relu"))

### Adding the output layer

In [ ]:
# 4. Add the final output layer (1 neuron because output is binary, Sigmoid for probability)
ann.add(tf.keras.layers.Dense(units=1, activation="sigmoid"))

## Part 3 - Training the ANN

### Compiling the ANN

In [ ]:
#Visualizing the Training Loop
#During each epoch, the model goes through a constant loop:
#Forward Pass: Takes 32 customer profiles and makes guesses.
#Loss Calculation: Compares guesses with true outcomes using binary_crossentropy.
#Backward Pass (Backpropagation): Sends the error back through the network.
#Optimization: The adam optimizer tweaks the internal weights slightly to minimize that error next time.

# 1. Compile the network (Configure the learning goals and optimizer)
# We use 'adam' to adjust weights, 'binary_crossentropy' for 0/1 errors, and track accuracy percentage.
ann.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

### Training the ANN on the Training set

In [ ]:
# 2. Train the network on the training data
# The model reads data in batches of 32 rows and loops through the entire dataset 100 times (epochs).
ann.fit(X_train, y_train, batch_size=32, epochs=100)

## Part 4 - Making the predictions and evaluating the model

### Predicting the result of a single observation

In [ ]:
# 1. Define the single customer data row inside a 2D array... 1,0,0 encode from 'France"
single_customer = [[1, 0, 0, 600, 1, 40, 3, 60000, 2, 1, 1, 50000]]

# 2. Scale the data using the training scaler
scaled_customer = sc.transform(single_customer)

# 3. Predict the probability of churn
raw_prediction = ann.predict(scaled_customer)

# 4. Convert probability to a True/False decision and print it
# Returns True if probability > 50%, otherwise returns False
final_decision = raw_prediction > 0.5
print(final_decision)

**Homework**

Use our ANN model to predict if the customer with the following informations will leave the bank:

Geography: France

Credit Score: 600

Gender: Male

Age: 40 years old

Tenure: 3 years

Balance: \$ 60000

Number of Products: 2

Does this customer have a credit card? Yes

Is this customer an Active Member: Yes

Estimated Salary: \$ 50000

So, should we say goodbye to that customer?

**Solution**

Therefore, our ANN model predicts that this customer stays in the bank!

**Important note 1:** Notice that the values of the features were all input in a double pair of square brackets. That's because the "predict" method always expects a 2D array as the format of its inputs. And putting our values into a double pair of square brackets makes the input exactly a 2D array.

**Important note 2:** Notice also that the "France" country was not input as a string in the last column but as "1, 0, 0" in the first three columns. That's because of course the predict method expects the one-hot-encoded values of the state, and as we see in the first row of the matrix of features X, "France" was encoded as "1, 0, 0". And be careful to include these values in the first three columns, because the dummy variables are always created in the first columns.

### Predicting the Test set results

In [ ]:
import numpy as np

# 1. Get raw probability predictions for all test cases
y_pred = ann.predict(X_test)

# 2. Turn probabilities into binary True/False (1 or 0) decisions
y_pred = y_pred > 0.5

# 3. Reshape both arrays into vertical columns and print them side-by-side
# Left Column = Predicted Values | Right Column = Actual Real Values
print(
    np.concatenate(
        (y_pred.reshape(len(y_pred), 1), y_test.reshape(len(y_test), 1)),
        axis=1,
    )
)

### Making the Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score

# 1. Generate the Confusion Matrix grid
# Compares true answers (y_test) with your model's guesses (y_pred)
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)

# 2. Calculate and print the overall accuracy percentage
final_accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy Score: {final_accuracy * 100:.2f}%")